# Single-qubit randomized benchmarking

Benchmark the average single-qubit Clifford fidelity over several random circuits.

In [ ]:
from pathlib import Path

from qblox_lab.config.hardware import create_hardware_agent
from qblox_lab.experiments.cal14_single_qubit_randomized_benchmarking import SingleQubitRandomizedBenchmarking

## Run parameters

In [ ]:
HARDWARE_CONFIG = Path(r"..\config\hw_config.json")
DEVICE_CONFIG = Path(r"..\config\dut_config.json")
FLUX_CONFIG = Path(r"..\config\flux_config.json")
OUTPUT_DIR = Path("data")
QUBITS = ["q0"]
CLIFFORD_LENGTHS = [0, 1, 2, 4, 8, 16, 32, 64, 96]
CIRCUITS = 20
REPETITIONS = 100
RANDOM_SEED = 7
READOUT_AMPLITUDE = None
DRIVE_OUTPUT_ATTENUATION = None
READOUT_OUTPUT_ATTENUATION = None
READOUT_INPUT_ATTENUATION = None
TIMEOUT = 600
CREATE_DUMMY_CONNECTIONS = False
SIMULATE = False
SIMULATION_DEPOLARIZING_PARAMETER = 0.99
SIMULATION_NOISE = None
PLOT_RESULTS = True

## Hardware and experiment

In [ ]:
hardware_agent = create_hardware_agent(
    hardware_configuration=HARDWARE_CONFIG, device_configuration=DEVICE_CONFIG,
    output_dir=OUTPUT_DIR, create_dummy_connections=CREATE_DUMMY_CONNECTIONS,
)
experiment = SingleQubitRandomizedBenchmarking(
    hardware_agent=hardware_agent, qubits=QUBITS, flux_config=FLUX_CONFIG,
)

## Measurement or simulation

The same seed reproduces the Clifford circuits. The acquisition-bin count is `length points * circuits * repetitions`.

In [ ]:
if SIMULATE:
    dataset = experiment.simulated_data(
        clifford_lengths=CLIFFORD_LENGTHS, circuits=CIRCUITS, repetitions=REPETITIONS,
        depolarizing_parameter=SIMULATION_DEPOLARIZING_PARAMETER,
        noise=SIMULATION_NOISE, seed=RANDOM_SEED,
    )
else:
    dataset = experiment.run_measurement(
        clifford_lengths=CLIFFORD_LENGTHS, circuits=CIRCUITS, repetitions=REPETITIONS,
        seed=RANDOM_SEED, readout_amplitude=READOUT_AMPLITUDE,
        drive_output_attenuation=DRIVE_OUTPUT_ATTENUATION,
        readout_output_attenuation=READOUT_OUTPUT_ATTENUATION,
        readout_input_attenuation=READOUT_INPUT_ATTENUATION, timeout=TIMEOUT,
    )
dataset

## Analysis

In [ ]:
results = experiment.analysis()
for qubit, result in results.items():
    if result.success:
        print(
            f"{qubit}: Clifford fidelity = {result.average_clifford_fidelity:.8f}, "
            f"EPC = {result.error_per_clifford:.3e}, "
            f"estimated primitive error = {result.estimated_error_per_primitive:.3e}"
        )
    else:
        print(f"{qubit}: randomized-benchmarking fit failed")
if PLOT_RESULTS:
    experiment.plot()

## Result persistence

RB reports gate performance but does not directly modify a calibrated device parameter.